# Single-signal parameter selection

Extracts causal signal-specific parameters, screens their incremental value, and selects compact sets. The registered test comparison is an archival evaluation step, not a tuning loop.

Run the setup cell first. Full processing and training require locally provisioned datasets; the grading quick start uses synthetic tests and saved aggregate results.


In [ ]:
from pathlib import Path
import sys, json, shutil, tempfile, time
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs' / 'hemodynamics.json').exists())
assert Path(sys.executable).resolve() == (ROOT / '.venv' / 'Scripts' / 'python.exe').resolve(), 'Select the project .venv kernel'
sys.path.insert(0, str(ROOT / 'notebooks'))
import importlib
import numpy as np, pandas as pd
from IPython.display import Image, Markdown, display
from hemodynamics import independent_features as F, independence_study as S
for _m in (F, S):
    importlib.reload(_m)
D, FIG = ROOT / 'results' / 'independence', ROOT / 'figures' / 'independence'
def table(name, rows=None, cols=None, round_=4):
    t = pd.read_csv(D / f'{name}.csv')
    t = t[cols] if cols else t
    display(t.round(round_) if rows is None else t.head(rows).round(round_))
def fig(name, note):
    display(Image(filename=str(FIG / f'{name}.png'))); display(Markdown(note))
pd.set_option('display.max_columns', 30); pd.set_option('display.width', 220); pd.set_option('display.max_colwidth', 120)

In [ ]:
SMOKE = False      # True = 30-case end-to-end regression run (extraction -> atlas -> feasibility screen) into a temp folder, < 5 min
WORKERS = 16

In [ ]:
if SMOKE:
    t0 = time.perf_counter()
    from hemodynamics.bshi import load_window_index
    idx = load_window_index(ROOT, 'ea_core', columns=['dataset', 'case_id'])
    rng = np.random.default_rng(5052)
    cases = [(ds, c) for ds in ('vitaldb', 'mover') for c in rng.choice(np.sort(idx.loc[idx.dataset == ds, 'case_id'].unique()), 15, replace=False)]
    name = 'window_params_smoke.parquet'
    tmp = Path(tempfile.mkdtemp(prefix='indep_smoke_'))
    try:
        table_, info, _ = F.build(ROOT, workers=WORKERS, cases=cases, write_native=False, out_name=name)
        print(f'extraction: {len(info)} cases, {int((info.error != "").sum())} failed, {len(table_)} windows [{time.perf_counter() - t0:.0f}s]')
        df = S.load_frame(ROOT, name)
        tr = S.training_subsample(df, step=1)
        for sub in (S.OUT, S.FIG):
            (tmp / sub).mkdir(parents=True, exist_ok=True)
        stab, clusters, orders = S.within_signal_atlas(tr, tmp)
        S.between_signal_atlas(tr, tmp, orders)
        S.temporal_before_ioh(tr, tmp, reps=20)
        pc = S.outcome_correlations(tr, tmp, reps=20)
        dom = S.domain_table(tr, tmp)
        print(f'atlas done [{time.perf_counter() - t0:.0f}s]')
        fp = tmp / 'screen_frame.parquet'
        tr.to_parquet(fp, index=False)
        t, ref = S.feasibility_screen(tr, tmp, fp, pc, dom, workers=4, reps=20, log=print)
        print(f'SMOKE OK: {len(t)} parameters screened, {int(t.feasible.sum())} feasible on 30 cases (not meaningful) [{time.perf_counter() - t0:.0f}s]')
    finally:
        for f in (ROOT / F.OUT_DIR / name, ROOT / F.OUT_DIR / 'window_params_smoke_cases.csv'):
            f.unlink(missing_ok=True)
        shutil.rmtree(tmp, ignore_errors=True)
summary = json.loads((D / 'summary.json').read_text(encoding='utf-8'))
print('plan sha256 (pre-registered):', summary['plan_sha256'][:16])

## Native single-signal extraction and quality checks
ART-native SBP/DBP/PP medians agree with the beat store within 0.25-0.44 mmHg (median |diff|); heart rate from ECG, ART and PPG alone agree with Spearman 0.998-1.000.

In [ ]:
q = pd.read_csv(D / 'table_qc.csv')
display(q[q.section.isin(['agreement with beat store / labels', 'heart-rate independence', 'failures', 'table'])].round(3))
cov = q[q.section == 'coverage']
display(cov.groupby('dataset')['pass'].agg(['sum', 'count']).rename(columns={'sum': 'params_meeting_F1_coverage', 'count': 'params'}))

## Correlation atlas (training split, every 3rd window)

In [ ]:
table('table_corr_structure_stability')
fig('fig_ind_01_within_signal_correlation', 'Within-signal redundancy is high (ART 88 parameters form 62 clusters at |rho| >= 0.8) and the ART and ECG correlation structure is stable across hospitals (0.91, 0.94); PPG is less stable (0.70).')
table('table_corr_between_named_pairs')
fig('fig_ind_02_between_signal_correlation', 'The three signals are not independent physiologically: rate is identical (rho 1.00), interval variability is shared (0.64-0.77), while ART and PPG amplitude variation only partly agree (0.25-0.50).')
fig('fig_ind_03_temporal_correlation_before_ioh', 'Beat-to-beat variance of MAP, SBP and PP is higher before IOH than in MAP-matched controls in both hospitals; lag-1 autocorrelation is higher only in VitalDB.')
fig('fig_ind_04_partial_correlation_vitaldb_vs_mover', 'Partial correlations with the future MAP drop given G0 agree in sign across hospitals for most single-signal parameters; the largest are 5-minute ART variability measures (about 0.18-0.25).')

## Feasibility screen (gates F1-F7)

In [ ]:
t = pd.read_csv(D / 'table_feasibility.csv')
display(t.groupby('signal')[['F1', 'F2', 'F3', 'F4', 'F5', 'F6', 'feasible']].sum())
display(t[t.feasible.astype(bool)][['param', 'family', 'lr_d_auroc', 'lr_d_auprc', 'hgb_d_auroc', 'hgb_d_auprc', 'transfer_d_auroc', 'domain_auroc']].round(4))
fig('fig_ind_05_feasibility_quadrant', 'Six of 147 parameters pass F1-F6; most single-signal parameters are hospital-agnostic but add too little over current MAP to survive the FDR-controlled incremental test.')
fig('fig_ind_06_incremental_value_forest', 'Incremental value over G0 is small for every feasible parameter (Delta AUROC at most about 0.005); the old cross-signal families are shown as grey reference rows.')

## Compact sets (training) and validation confirmation; frozen in `configs/hemodynamics.json` key `independence`

In [ ]:
print(json.dumps(json.loads((D / 'frozen_independence.json').read_text(encoding='utf-8'))['sets'], indent=1))
table('table_validation_confirm', cols=None, rows=40)
fig('fig_ind_07_selection_path', 'Greedy forward selection stops after 1-3 parameters per set; IND-EA = ECG SDNN (300 s) + ART MAP slope (60 s) + ART dP/dt min (60 s), and no PPG parameter is added (IND-EAP = IND-EA).')

## Final comparison on the test split (used once)

In [ ]:
r = pd.read_csv(D / 'table_old_vs_new.csv')
keep = ['pooled_test', 'E3x vitaldb_train->all_mover (not clean)', 'clean vitaldb_train->mover_test', 'reverse mover_train->vitaldb_test (not clean)']
h = r[(r.learner == 'hgb') & r.setting.isin(keep) & r.model.isin(['G0', 'old G2', 'old G4', 'G0 + old cross arm', 'G0 + IND-EA', 'IND-EA alone'])]
display(h[['setting', 'window_set', 'model', 'auroc', 'auroc_lo', 'auroc_hi', 'auprc', 'auprc_lo', 'auprc_hi']].round(3).head(40))
table('table_criteria')
fig('fig_ind_08_old_vs_new', 'G0 + IND-EA (3 independent parameters) beats current MAP on the pooled test (Delta AUROC +0.011) and, unlike the old sets, does not lose performance when transferred to MOVER.')
fig('fig_ind_09_lead_time_and_gray_zone', 'The gain over G0 grows with lead time (+0.016 to +0.020 AUROC at 120-240 s) and is concentrated outside the gray zone.')
table('table_domain_old_vs_new')
fig('fig_ind_10_domain_shift_old_vs_new', 'Old feature sets identify the hospital strongly (domain AUROC 0.82-1.00); IND-A is hospital-agnostic (0.51), and the ECG SDNN term is what makes IND-EA partly identifiable (0.78).')
fig('fig_ind_11_example_cases', 'Illustration only: MAP and the three IND-EA parameters over the 10 minutes before a VitalDB and a MOVER training event, with both alarm thresholds.')

## PPG decision

In [ ]:
table('table_ppg_decision', rows=40)
fig('fig_ind_12_ppg_decision', 'PPG adds nothing over IND-EA, requiring it removes 49% of MOVER events, and the noninvasive ECG + PPG set predicts ART-defined IOH poorly and is hospital-specific: PPG is removed from the main model.')